# Session 2: Models in Computer Networking: OSI and TCP/IP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-02-osi-and-tcpip-models.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-02_encapsulate.py`

In [1]:
"""Session 2 - Encapsulation with real header bytes: Ethernet / IPv4 / TCP.

Builds an HTTP request the way a host's protocol stack does, layer by layer,
computes the real IPv4 header checksum, prints a hex dump, then
de-encapsulates it the way the receiver does. All addresses are lab addresses.
Run:  python session-02_encapsulate.py
"""
import struct


def checksum(data: bytes) -> int:
    """Internet checksum (RFC 1071): one's complement of the one's-complement sum."""
    if len(data) % 2:
        data += b"\0"
    s = sum(struct.unpack(f"!{len(data)//2}H", data))
    while s >> 16:
        s = (s & 0xFFFF) + (s >> 16)
    return ~s & 0xFFFF


def mac(s): return bytes(int(x, 16) for x in s.split(":"))
def ip(s): return bytes(int(x) for x in s.split("."))


# Layer 7: application data
payload = b"GET / HTTP/1.1\r\nHost: lab.local\r\n\r\n"

# Layer 4: TCP header (20 bytes, no options). Flags 0x18 = PSH + ACK
tcp = struct.pack("!HHIIBBHHH", 50123, 80, 1000, 2000, 5 << 4, 0x18, 64240, 0, 0)
segment = tcp + payload

# Layer 3: IPv4 header (20 bytes). Checksum computed over the header only
ver_ihl, tos, total_len = 0x45, 0, 20 + len(segment)
ident, flags_frag, ttl, proto = 0x1C46, 0x4000, 64, 6
hdr = struct.pack("!BBHHHBBH4s4s", ver_ihl, tos, total_len, ident, flags_frag, ttl, proto, 0,
                  ip("192.168.10.25"), ip("192.168.20.80"))
hdr = hdr[:10] + struct.pack("!H", checksum(hdr)) + hdr[12:]
packet = hdr + segment

# Layer 2: Ethernet II header (dst MAC, src MAC, EtherType 0x0800 = IPv4)
frame = mac("00:1a:2b:3c:4d:5e") + mac("00:50:56:aa:bb:cc") + b"\x08\x00" + packet


def hexdump(b, width=16):
    for i in range(0, len(b), width):
        chunk = b[i:i + width]
        text = "".join(chr(c) if 32 <= c < 127 else "." for c in chunk)
        print(f"  {i:04x}  {chunk.hex(' '):<{width*3}} {text}")


if __name__ == "__main__":
    print(f"L7 data     {len(payload):3} bytes")
    print(f"L4 segment  {len(segment):3} bytes (+20 TCP header)")
    print(f"L3 packet   {len(packet):3} bytes (+20 IPv4 header)")
    print(f"L2 frame    {len(frame):3} bytes (+14 Ethernet header, FCS added by the NIC)\n")
    hexdump(frame)

    print("\nReceiver de-encapsulates:")
    dst, src, etype = frame[0:6], frame[6:12], struct.unpack("!H", frame[12:14])[0]
    print(f"  L2: dst {dst.hex(':')} src {src.hex(':')} type 0x{etype:04x} -> IPv4")
    ih = frame[14:34]
    v, _, tl, _, _, t, pr, cs, s, d = struct.unpack("!BBHHHBBH4s4s", ih)
    print(f"  L3: version {v >> 4}, length {tl}, TTL {t}, protocol {pr} (TCP), "
          f"{'.'.join(map(str, s))} -> {'.'.join(map(str, d))}")
    print(f"      header checksum 0x{cs:04x}; recomputed over header = 0x{checksum(ih):04x} (0 means valid)")
    sp, dp, seq, ack = struct.unpack("!HHII", frame[34:46])
    print(f"  L4: port {sp} -> {dp} (HTTP), seq {seq}, ack {ack}")
    first_line = frame[54:].split(b"\r\n")[0].decode()
    print(f"  L7: {first_line!r} (application data)")

L7 data      35 bytes
L4 segment   55 bytes (+20 TCP header)
L3 packet    75 bytes (+20 IPv4 header)
L2 frame     89 bytes (+14 Ethernet header, FCS added by the NIC)

  0000  00 1a 2b 3c 4d 5e 00 50 56 aa bb cc 08 00 45 00  ..+<M^.PV.....E.
  0010  00 4b 1c 46 40 00 40 06 7e ad c0 a8 0a 19 c0 a8  .K.F@.@.~.......
  0020  14 50 c3 cb 00 50 00 00 03 e8 00 00 07 d0 50 18  .P...P........P.
  0030  fa f0 00 00 00 00 47 45 54 20 2f 20 48 54 54 50  ......GET / HTTP
  0040  2f 31 2e 31 0d 0a 48 6f 73 74 3a 20 6c 61 62 2e  /1.1..Host: lab.
  0050  6c 6f 63 61 6c 0d 0a 0d 0a                       local....

Receiver de-encapsulates:
  L2: dst 00:1a:2b:3c:4d:5e src 00:50:56:aa:bb:cc type 0x0800 -> IPv4
  L3: version 4, length 75, TTL 64, protocol 6 (TCP), 192.168.10.25 -> 192.168.20.80
      header checksum 0x7ead; recomputed over header = 0x0000 (0 means valid)
  L4: port 50123 -> 80 (HTTP), seq 1000, ack 2000
  L7: 'GET / HTTP/1.1' (application data)


## `session-02_osi_mapper.py`

In [2]:
"""Session 2 - Map protocols, devices and PDUs to OSI and TCP/IP layers.

Also classifies example networks into PAN / LAN / CAN / MAN / WAN by their
span, which is the "types of area networks" part of the session.
Run:  python session-02_osi_mapper.py
"""

OSI = {7: "Application", 6: "Presentation", 5: "Session", 4: "Transport",
       3: "Network", 2: "Data Link", 1: "Physical"}
TCPIP = {7: "Application", 6: "Application", 5: "Application", 4: "Transport",
         3: "Internet", 2: "Network Access", 1: "Network Access"}
PDU = {7: "Data", 6: "Data", 5: "Data", 4: "Segment / Datagram", 3: "Packet", 2: "Frame", 1: "Bits"}

ITEMS = {"HTTP": 7, "DNS": 7, "SSH": 7, "TLS encryption": 6, "JPEG / ASCII": 6,
         "NetBIOS / RPC sessions": 5, "TCP": 4, "UDP": 4, "IPv4 / IPv6": 3, "ICMP (ping)": 3,
         "OSPF": 3, "Router": 3, "Ethernet / 802.3": 2, "Wi-Fi / 802.11": 2, "ARP": 2,
         "Switch": 2, "Hub / repeater": 1, "Cat6 cable / fibre": 1}


def area_type(span_m):
    if span_m <= 10: return "PAN"
    if span_m <= 1_000: return "LAN"
    if span_m <= 5_000: return "CAN"
    if span_m <= 50_000: return "MAN"
    return "WAN"


if __name__ == "__main__":
    print(f"{'Item':24} {'OSI layer':18} {'TCP/IP layer':15} PDU")
    for item, layer in ITEMS.items():
        print(f"{item:24} {str(layer) + ' ' + OSI[layer]:18} {TCPIP[layer]:15} {PDU[layer]}")
    print("\nArea network types by span")
    for name, span in [("Bluetooth earbuds to phone", 3), ("Computer lab", 60),
                       ("University campus, 6 buildings", 2_500), ("City-wide bank branches", 30_000),
                       ("Offices in Karachi and Islamabad", 1_150_000)]:
        print(f"  {name:36} {span:>9,} m  -> {area_type(span)}")

Item                     OSI layer          TCP/IP layer    PDU
HTTP                     7 Application      Application     Data
DNS                      7 Application      Application     Data
SSH                      7 Application      Application     Data
TLS encryption           6 Presentation     Application     Data
JPEG / ASCII             6 Presentation     Application     Data
NetBIOS / RPC sessions   5 Session          Application     Data
TCP                      4 Transport        Transport       Segment / Datagram
UDP                      4 Transport        Transport       Segment / Datagram
IPv4 / IPv6              3 Network          Internet        Packet
ICMP (ping)              3 Network          Internet        Packet
OSPF                     3 Network          Internet        Packet
Router                   3 Network          Internet        Packet
Ethernet / 802.3         2 Data Link        Network Access  Frame
Wi-Fi / 802.11           2 Data Link        Network Ac